# LIP-H0-017 — paired closed-loop screen

Notebook operacional mínimo, reconstruído em 2026-08-26 a partir dos contratos congelados.

- Commit obrigatório: `25472732a7606fccf54d21b20c97295da3fb2894`.
- GPU obrigatória: NVIDIA L4.
- Sem `drive.mount()`: entradas são lidas pela API do Drive e gravadas apenas em `/content/lip-cache`.
- Ordem causal: preflight → controle `open_loop_zero_live` → tratamento `closed_loop_live` → agregação.
- Split de confirmação é proibido; um eventual passe autoriza somente o desenho do EVAL-038.
- Baixe o pacote de cada braço antes de avançar, porque o armazenamento do runtime é efêmero.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/zigfreud/latent-llm-agent-communication.git"
BRANCH = "exp/LIP-H0-017-closed-loop-trajectory"
EXPECTED_COMMIT = "25472732a7606fccf54d21b20c97295da3fb2894"
REPO = Path("/content/latent-llm-agent-communication")
CACHE = Path("/content/lip-cache")
INPUTS = CACHE / "inputs"
RESULTS = CACHE / "results/LIP-H0-017/screen-v1"
for path in (CACHE, INPUTS, RESULTS):
    path.mkdir(parents=True, exist_ok=True)

def run(args, *, cwd=None):
    print("+", " ".join(map(str, args)), flush=True)
    return subprocess.run(args, cwd=cwd, check=True, text=True)

if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(f"Refusing non-git path: {REPO}")
if not (REPO / ".git").is_dir():
    run(["git", "clone", REPO_URL, str(REPO)])
if subprocess.run(["git", "diff", "--quiet"], cwd=REPO).returncode != 0:
    raise RuntimeError("Tracked Colab repository changes detected")
if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO).returncode != 0:
    raise RuntimeError("Staged Colab repository changes detected")
run(["git", "fetch", "origin", BRANCH], cwd=REPO)
local_exists = subprocess.run(["git", "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"], cwd=REPO).returncode == 0
if local_exists:
    run(["git", "switch", BRANCH], cwd=REPO)
else:
    run(["git", "switch", "--track", "-c", BRANCH, f"origin/{BRANCH}"], cwd=REPO)
run(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO)
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert HEAD == EXPECTED_COMMIT, (HEAD, EXPECTED_COMMIT)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-protocol.txt"], cwd=REPO)
print({"commit": HEAD, "cache": str(CACHE), "drive_mounted_or_used": False})

In [ ]:
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import hashlib, io, json, uuid

auth.authenticate_user()
credentials, _ = default()
drive_api = build("drive", "v3", credentials=credentials, cache_discovery=False)
FOLDER_MIME = "application/vnd.google-apps.folder"

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def drive_meta(file_id):
    return drive_api.files().get(fileId=file_id, fields="id,name,mimeType,size", supportsAllDrives=True).execute()

def list_children(folder_id):
    page_token = None
    rows = []
    while True:
        response = drive_api.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType,size)",
            pageSize=1000, pageToken=page_token,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        rows.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            return sorted(rows, key=lambda row: row["name"])

def download_file(item, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(item.get("size", 0))
    if destination.is_file() and expected_size and destination.stat().st_size == expected_size:
        print("REUSED", destination)
        return destination
    if destination.exists():
        raise RuntimeError(f"Existing input has unexpected size; inspect manually: {destination}")
    temporary = destination.with_name(destination.name + ".part-" + uuid.uuid4().hex)
    request = drive_api.files().get_media(fileId=item["id"], supportsAllDrives=True)
    with io.FileIO(temporary, "wb") as handle:
        downloader = MediaIoBaseDownload(handle, request, chunksize=32 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"{destination.name}: {status.progress() * 100:.1f}%", flush=True)
    temporary.replace(destination)
    if expected_size and destination.stat().st_size != expected_size:
        raise RuntimeError(f"Size mismatch: {destination}")
    return destination

def download_folder(folder_id, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for item in list_children(folder_id):
        target = destination / item["name"]
        if item["mimeType"] == FOLDER_MIME:
            download_folder(item["id"], target)
        else:
            download_file(item, target)

IDS = {
    "packet_bundle_folder": "18Fe8f9s0PN8w-Ert8vUnC7C6Hc4yOLV6",
    "source_checkpoint": "1iiAeXtSY8ppYLdOUSLJnZo2WyGuILjH-",
    "candidate_bank": "1nVSWgBrco2KE2fyUd-SsdbXh_6JXoe4q",
    "pilot_summary": "19pUee8c7zDYABmw8DepARvJuEIVLHsf9",
}
BUNDLE = INPUTS / "packet-bundle"
SOURCE_CHECKPOINT = INPUTS / "H0-015_best_checkpoint.pt"
CANDIDATE_BANK = INPUTS / "candidate_banks.json"
PILOT_SUMMARY = INPUTS / "H0-017_pilot_run_summary.json"
download_folder(IDS["packet_bundle_folder"], BUNDLE)
for key, target in (("source_checkpoint", SOURCE_CHECKPOINT), ("candidate_bank", CANDIDATE_BANK), ("pilot_summary", PILOT_SUMMARY)):
    download_file(drive_meta(IDS[key]), target)
assert sha256_path(BUNDLE / "manifest.json") == "ad501e9537283b26f3b889ed7f355f69b47dd2e7967e6a92554a70ca5a7a3744"
assert sha256_path(SOURCE_CHECKPOINT) == "f5a6435dee9df6a16204be8b6c00dc3e2438222ee99dcc30018c9f822fbeb405"
assert sha256_path(CANDIDATE_BANK) == "ca4cb9cd7aec2981b76227265dbe325c77be00dba145014a83f1bf0ca2def7e4"
print({"bundle_shards": len(list((BUNDLE / "shards").glob("*.pt"))), "inputs_bytes": sum(p.stat().st_size for p in INPUTS.rglob("*") if p.is_file())})

In [ ]:
import json, torch
sys.path.insert(0, str(REPO))
from src.core.packet_bundle import validate_packet_bundle

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable")
GPU_NAME = torch.cuda.get_device_name(0)
if "L4" not in GPU_NAME.upper():
    raise RuntimeError(f"Frozen contract requires L4; observed {GPU_NAME}")
validation = validate_packet_bundle(BUNDLE, require_real=True)
assert validation["split_counts"] == {"train": 256, "development_selection": 32, "development_gate": 32, "confirmation": 0}
pilot = json.loads(PILOT_SUMMARY.read_text(encoding="utf-8"))
assert pilot["stage"] == "pilot" and pilot["variant"] == "closed_loop_live"
assert pilot["complete"] is True and pilot["pilot_gate"]["passed"] is True
assert pilot["run_commit"] == "5ccc26098ffa226a0969ed9efd82a6b6ecf890ac"
run([sys.executable, "-m", "src.scripts.run_closed_loop_trajectory", "--dry-run-contract"], cwd=REPO)
print({"gpu": GPU_NAME, "cuda": torch.version.cuda, "commit": HEAD, "bundle": validation, "pilot_passed": True})

In [ ]:
from google.colab import files
import shutil

def run_arm(variant):
    output_dir = RESULTS / variant
    summary = output_dir / "run_summary.json"
    if output_dir.exists() and any(output_dir.iterdir()) and not summary.is_file():
        raise RuntimeError(f"Partial output requires manual inspection: {output_dir}")
    if not summary.is_file():
        run([
            sys.executable, "-m", "src.scripts.run_closed_loop_trajectory",
            "--bundle-dir", str(BUNDLE),
            "--source-checkpoint", str(SOURCE_CHECKPOINT),
            "--candidate-bank", str(CANDIDATE_BANK),
            "--output-dir", str(output_dir),
            "--variant", variant,
            "--target-device", "cuda",
        ], cwd=REPO)
    payload = json.loads(summary.read_text(encoding="utf-8"))
    assert payload["complete"] is True and payload["stage"] == "paired_screen_cell"
    assert payload["variant"] == variant and payload["training"]["updates_completed"] == 128
    assert payload["run_commit"] == EXPECTED_COMMIT
    archive = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
    files.download(archive)
    return payload, Path(archive)

CONTROL, CONTROL_ARCHIVE = run_arm("open_loop_zero_live")
print({"variant": CONTROL["variant"], "updates": CONTROL["training"]["updates_completed"], "summary": CONTROL["development_gate_metrics"]["incoming_trajectory"]["normalized_residual_rmse"], "archive": str(CONTROL_ARCHIVE)})

In [ ]:
TREATMENT, TREATMENT_ARCHIVE = run_arm("closed_loop_live")
print({"variant": TREATMENT["variant"], "updates": TREATMENT["training"]["updates_completed"], "summary": TREATMENT["development_gate_metrics"]["incoming_trajectory"]["normalized_residual_rmse"], "archive": str(TREATMENT_ARCHIVE)})

In [ ]:
AGGREGATE = RESULTS / "paired_aggregate.json"
run([
    sys.executable, "-m", "src.scripts.aggregate_closed_loop_screen",
    "--pilot-summary", str(PILOT_SUMMARY),
    "--control-summary", str(RESULTS / "open_loop_zero_live/run_summary.json"),
    "--treatment-summary", str(RESULTS / "closed_loop_live/run_summary.json"),
    "--output", str(AGGREGATE),
], cwd=REPO)
aggregate = json.loads(AGGREGATE.read_text(encoding="utf-8"))
print(json.dumps({"aggregate_gate": aggregate["aggregate_gate"], "decision": aggregate["decision"]}, indent=2))
FINAL_ARCHIVE = shutil.make_archive(str(CACHE / "LIP-H0-017_screen-v1_complete"), "zip", root_dir=RESULTS)
files.download(FINAL_ARCHIVE)
print({"final_archive": FINAL_ARCHIVE, "sha256": sha256_path(FINAL_ARCHIVE)})